# Nemotron LoRA Training — Fully Offline, RTX Pro 6000

**Required Kaggle inputs (add in sidebar before running):**
- `nemotron-offline-deps` dataset — contains wheelhouse (129 wheels) + NuminaMath parquet
- `huikang/nemotron-adapter` — competition LoRA adapter
- `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16` — base model (separate model input)

**Internet must be OFF.**

---

### All fixes applied vs original notebook

| # | Cell | Bug | Fix |
|---|------|-----|-----|
| 1 | 1 | Duplicate discover cell re-ran `find_wheelhouse`, imported unused `packaging.version.Version` | Merged into single install cell |
| 2 | 1 | Stray wheel-list cell used `os` without importing → `NameError` on fresh kernel | Folded into Cell 1 |
| 3 | 2 | `RuntimeError` message referenced wrong model name (`30B`) | Corrected to `4B` |
| 4 | 2 | `BASE_MODEL_PATH`/`ADAPTER_PATH` stayed `None` when path type unknown → silent crash | Added `else: raise RuntimeError` |
| 5 | 3 | `PeftModel.from_pretrained` on raw 4-bit model → bitsandbytes gradient errors | Use `FastLanguageModel.get_peft_model` first, then `load_adapter` |
| 6 | 6 | `eval_strategy=` rejected by transformers ≥4.46 | Changed to `evaluation_strategy=` |
| 7 | 6 | `SFTTrainer(tokenizer=…)` rejected by trl ≥0.9 | Changed to `processing_class=tokenizer` |
| 8 | 7 | `GRPOConfig(max_new_tokens=…)` not a valid arg | Changed to `max_completion_length=` |
| 9 | 7 | `GRPOConfig(temperature=…)` not a valid top-level arg | Moved to `generation_kwargs=` |
| 10 | 7 | `reward_correctness(…, solution, …)` crashes if column absent | Added `solution=None` default + guard |
| 11 | 1 | `sklearn → scipy → numpy` import chain crashed: numpy on disk (2.4.4) ≠ in memory (2.0.2) | Remove sklearn; write `constraints.txt` pinning numpy to in-memory version so pip never upgrades it |
| 12 | 1 | `unsloth` imported after `transformers` → patches not applied, `UserWarning` | Install + import `unsloth_zoo`/`unsloth` before all other packages |
| 13 | 1 | `os._exit(0)` used for restart → hard-kills kernel process → Kaggle shows "Kernel Restarting" crash | Removed all restart logic; numpy pin + sklearn removal make restart unnecessary |


## Cell 1 — Install all packages from local wheels (offline)

In [1]:
import os, glob, json, subprocess, sys
import importlib.metadata as imd

# ── locate wheelhouse ─────────────────────────────────────────────────────────
def find_wheelhouse():
    for dirpath, _, files in os.walk('/kaggle/input'):
        if any(f.endswith('.whl') for f in files):
            return dirpath
    return None

WHEEL_DIR = find_wheelhouse()
assert WHEEL_DIR, 'Wheelhouse not found — attach nemotron-offline-deps dataset'
print(f'Wheelhouse: {WHEEL_DIR}')

with open(os.path.join(WHEEL_DIR, 'manifest.json')) as f:
    manifest = json.load(f)
print('manifest:', json.dumps(manifest, indent=2))

all_wheels = sorted(glob.glob(WHEEL_DIR + '/*.whl'))
print(f'\nFound {len(all_wheels)} wheels:')
for w in all_wheels:
    print(f'  {os.path.basename(w)}')

# ── FIX 11: pin numpy to whatever is already in memory ───────────────────────
# unsloth_zoo hard-checks: if numpy.__version__ (in memory) != installed numpy
# (on disk) → RuntimeError. Kaggle boots with numpy 2.0.2; the wheelhouse has
# 2.4.4. If pip upgrades it, unsloth_zoo fails immediately on import.
# Solution: write a pip constraints file locking numpy to the current in-memory
# version so no package can pull in the upgrade as a dependency.
import numpy as _np
MEM_NUMPY = _np.__version__
print(f'\nnumpy in memory : {MEM_NUMPY}  (will be pinned throughout install)')

CONSTRAINTS = '/tmp/numpy_pin.txt'
with open(CONSTRAINTS, 'w') as f:
    f.write(f'numpy=={MEM_NUMPY}\n')

def install(spec, no_deps=False, force=False):
    cmd = [sys.executable, '-m', 'pip', 'install',
           '--no-index', f'--find-links={WHEEL_DIR}',
           '--constraint', CONSTRAINTS,  # keeps numpy frozen
           spec, '-q']
    if no_deps:
        cmd.append('--no-deps')
    if force:
        cmd.append('--force-reinstall')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ERROR {spec}: {r.stderr.strip()[:300]}')
        return False
    print(f'  OK: {spec}')
    return True

# ── FIX 11 (cont): remove sklearn before installing anything ─────────────────
# transformers.candidate_generator conditionally does:
#   if is_sklearn_available(): from sklearn.metrics import roc_curve
# That chain is: sklearn → scipy → array_api_compat → `from numpy import *`
# which explodes when disk-numpy ≠ memory-numpy.
# Removing sklearn makes is_sklearn_available() return False → chain never runs.
# mamba_ssm has no sklearn dependency so this is safe.
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', 'scikit-learn', '-y', '-q'],
    capture_output=True, text=True)
print(f'sklearn: {"removed" if r.returncode == 0 else "already absent (OK)"}')

print('\nInstalling packages...')

# ── FIX 12: install unsloth/unsloth_zoo BEFORE transformers ──────────────────
# unsloth must be imported before transformers to patch it at load time.
# Install order: unsloth_zoo → unsloth → rest → transformers pinned last.
unsloth_zoo_ver = manifest.get('unsloth_zoo', '')
install(f'unsloth_zoo=={unsloth_zoo_ver}' if unsloth_zoo_ver else 'unsloth_zoo')

unsloth_ver = manifest.get('unsloth', '')
install(f'unsloth=={unsloth_ver}' if unsloth_ver else 'unsloth', no_deps=True)

for pkg in ['huggingface_hub', 'bitsandbytes', 'accelerate', 'datasets', 'peft']:
    ver = manifest.get(pkg.lower(), '')
    install(f'{pkg}=={ver}' if ver else pkg)

install('trl', no_deps=True)

tf_ver = manifest.get('transformers', '5.5.0')
install(f'transformers=={tf_ver}', force=True)

# mamba_ssm / causal_conv1d — wheel name contains CUDA tag, use glob
for pattern, pip_name in [('causal_conv1d*.whl', 'causal-conv1d'),
                           ('mamba_ssm*.whl',     'mamba-ssm')]:
    matches = glob.glob(os.path.join(WHEEL_DIR, pattern))
    if matches:
        print(f'  Installing {pip_name}: {os.path.basename(matches[0])}')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install',
             matches[0], '--no-deps', '--no-index', '--force-reinstall', '-q'],
            capture_output=True, text=True)
        print(f'  {"OK" if r.returncode == 0 else "ERROR: " + r.stderr[:300]}')
    else:
        print(f'  WARNING: no wheel found for {pip_name}')

# ── verify numpy was not touched ──────────────────────────────────────────────
disk_numpy = imd.version('numpy')
print(f'\nnumpy in memory : {MEM_NUMPY}')
print(f'numpy on disk   : {disk_numpy}')
if disk_numpy != MEM_NUMPY:
    raise RuntimeError(
        f'numpy was upgraded despite constraints file!\n'
        f'  memory={MEM_NUMPY}  disk={disk_numpy}\n'
        f'Identify which package pulled it in and add it to constraints.txt.'
    )
print('✓ numpy versions match — unsloth_zoo check will pass')

# ── versions summary ──────────────────────────────────────────────────────────
print('\nFinal versions:')
for pkg in ['numpy', 'unsloth', 'unsloth_zoo', 'trl', 'peft',
            'transformers', 'bitsandbytes', 'mamba_ssm', 'causal_conv1d']:
    try:
        print(f'  {pkg}: {imd.version(pkg)}')
    except Exception:
        print(f'  {pkg}: NOT FOUND')

# Canary imports — if numpy pin and sklearn removal both worked these succeed
import unsloth                    # FIX 12: must come before transformers
import mamba_ssm, causal_conv1d
print(f'\n✓ unsloth       : {unsloth.__version__}')
print(f'✓ mamba_ssm     : {mamba_ssm.__version__}')
print(f'✓ causal_conv1d : {causal_conv1d.__version__}')
print('\nCell 1 complete — proceed to Cell 2.')


Wheelhouse: /kaggle/input/datasets/prajeeta/nemotron-prep-prats/wheelhouse
manifest: {
  "transformers": "5.5.0",
  "unsloth": "2026.5.2",
  "unsloth_zoo": "2026.5.1",
  "trl": "0.24.0",
  "peft": "0.18.1",
  "bitsandbytes": "0.49.2",
  "accelerate": "1.12.0",
  "datasets": "4.3.0",
  "huggingface_hub": "1.14.0",
  "causal_conv1d": "1.6.1",
  "mamba_ssm": "2.3.1"
}

Found 129 wheels:
  accelerate-1.13.0-py3-none-any.whl
  aiohappyeyeballs-2.6.1-py3-none-any.whl
  aiohttp-3.13.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
  aiosignal-1.4.0-py3-none-any.whl
  annotated_doc-0.0.4-py3-none-any.whl
  annotated_types-0.7.0-py3-none-any.whl
  anyio-4.13.0-py3-none-any.whl
  attrs-26.1.0-py3-none-any.whl
  bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
  causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
  certifi-2026.4.22-py3-none-any.whl
  charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.many

## Cell 2 — Detect model type & locate base model

Inspects the competition path to determine whether it is a LoRA adapter or a
base model, finds the real base model from your separately-attached Kaggle model
input, and patches `adapter_config.json` if `base_model_name_or_path` is null.

**Fixes:** corrected wrong model name in error (`30B` → `4B`);
added `else: raise` so an unrecognised path type fails immediately instead of
leaving `BASE_MODEL_PATH = None` and crashing silently later.


In [2]:
import os, json, shutil

COMPETITION_PATH = '/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20'

files_here = os.listdir(COMPETITION_PATH) if os.path.isdir(COMPETITION_PATH) else []
print('Files at competition path:')
for f in sorted(files_here):
    print(f'  {f}')

IS_ADAPTER = 'adapter_config.json' in files_here
IS_BASE    = 'config.json' in files_here and not IS_ADAPTER
print(f'\nPath type: {"LoRA adapter" if IS_ADAPTER else "base model" if IS_BASE else "UNKNOWN"}')

BASE_MODEL_PATH = None
ADAPTER_PATH    = None

if IS_BASE:
    BASE_MODEL_PATH = COMPETITION_PATH
    print('Competition path is a base model — using directly.')

elif IS_ADAPTER:
    with open(os.path.join(COMPETITION_PATH, 'adapter_config.json')) as f:
        adapter_cfg = json.load(f)
    print('\nadapter_config.json:')
    print(json.dumps(adapter_cfg, indent=2))

    # Find the real base model: has config.json + weights, no adapter_config.json
    print('\nSearching for base model in /kaggle/input...')
    candidates = []
    for dirpath, _, dirfiles in os.walk('/kaggle/input'):
        if COMPETITION_PATH in dirpath:
            continue
        if 'config.json' in dirfiles and 'adapter_config.json' not in dirfiles:
            has_weights = any(
                f.endswith('.safetensors') or f.endswith('.bin') for f in dirfiles
            )
            if has_weights:
                try:
                    with open(os.path.join(dirpath, 'config.json')) as f:
                        cfg = json.load(f)
                    candidates.append((dirpath, cfg.get('model_type', '?')))
                except Exception:
                    pass

    print(f'Candidates found: {len(candidates)}')
    for p, mt in candidates:
        print(f'  [{mt}] {p}')

    if not candidates:
        raise RuntimeError(
            'No base model found in /kaggle/input!\n'
            'Add nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16 as a Kaggle model input.'
        )

    BASE_MODEL_PATH = candidates[0][0]
    print(f'\nSelected base model: {BASE_MODEL_PATH}')

    # Patch adapter_config if base_model_name_or_path is null/missing
    current_ref = adapter_cfg.get('base_model_name_or_path')
    if not current_ref or str(current_ref).lower() in ('none', 'null', ''):
        print(f'Patching adapter_config: base_model_name_or_path was {current_ref!r}')
        ADAPTER_COPY = '/kaggle/working/adapter_patched'
        if os.path.exists(ADAPTER_COPY):
            shutil.rmtree(ADAPTER_COPY)
        shutil.copytree(COMPETITION_PATH, ADAPTER_COPY)
        patched = dict(adapter_cfg)
        patched['base_model_name_or_path'] = BASE_MODEL_PATH
        with open(os.path.join(ADAPTER_COPY, 'adapter_config.json'), 'w') as f:
            json.dump(patched, f, indent=2)
        ADAPTER_PATH = ADAPTER_COPY
        print(f'Patched adapter written to: {ADAPTER_PATH}')
    else:
        print(f'base_model_name_or_path already set: {current_ref}')
        ADAPTER_PATH = COMPETITION_PATH

else:
    # FIX 4: was silently leaving BASE_MODEL_PATH=None → crash in Cell 3
    raise RuntimeError(
        f'Competition path is neither a base model nor a LoRA adapter.\n'
        f'Path: {COMPETITION_PATH}\n'
        f'Files found: {files_here}'
    )

print(f'\nBASE_MODEL_PATH = {BASE_MODEL_PATH}')
print(f'ADAPTER_PATH    = {ADAPTER_PATH}')


Files at competition path:
  README.md
  adapter_config.json
  adapter_model.safetensors
  checkpoint_complete

Path type: LoRA adapter

adapter_config.json:
{
  "alpha_pattern": {},
  "auto_mapping": null,
  "base_model_name_or_path": null,
  "bias": "none",
  "corda_config": null,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": false,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
  "r": 32,
  "rank_pattern": {},
  "revision": null,
  "target_modules": "all-linear",
  "task_type": "CAUSAL_LM",
  "trainable_token_indices": null,
  "use_dora": false,
  "use_rslora": false
}

Searching for base model in /kaggle/input...
Candidates found: 1
  [nemotron_h] /kaggle/input/models/met

## Cell 3 — Load base model + competition adapter (4-bit QLoRA)

**Fix 5:** The original called `PeftModel.from_pretrained()` directly on a
4-bit quantised model, bypassing `prepare_model_for_kbit_training` and causing
bitsandbytes gradient errors at training time. The correct unsloth pattern:
1. Load base with `FastLanguageModel.from_pretrained` (handles 4-bit setup)
2. Wrap with `FastLanguageModel.get_peft_model` (prepares kbit training)
3. Load saved adapter weights with `model.load_adapter` on the already-wrapped model


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
LORA_RANK   = 32

# 1. Load base model
print(f'Loading base model: {BASE_MODEL_PATH}')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    trust_remote_code=True,
)
print(f'Base loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

# 2. Wrap with get_peft_model — this prepares the model for kbit training
#    FIX 5: original skipped this step and called PeftModel.from_pretrained
#    on the raw quantised model → bitsandbytes gradient errors at train time
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=64,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA applied.')

# 3. Load competition adapter weights into the already-wrapped model
if ADAPTER_PATH:
    print(f'\nLoading competition adapter: {ADAPTER_PATH}')
    model.load_adapter(ADAPTER_PATH, adapter_name='competition', is_trainable=True)
    model.set_adapter('competition')
    print('Competition adapter loaded — continuing training from checkpoint.')
else:
    print('No adapter path — training from scratch with fresh LoRA.')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'\nTrainable : {trainable / 1e6:.1f} M  ({100 * trainable / total:.2f}%)')
print(f'VRAM      : {torch.cuda.memory_allocated() / 1e9:.1f} GB')


Loading base model: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.5.2: Fast Nemotron_H patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

## Cell 4 — Load NuminaMath dataset from local parquet (offline)

In [ ]:
from datasets import load_dataset
import os

def find_parquet(hint='numinamath'):
    # Search by directory/file name hint first
    for dirpath, _, files in os.walk('/kaggle/input'):
        if hint.lower() in dirpath.lower() or any(hint in f for f in files):
            for f in files:
                if f.endswith('.parquet'):
                    return os.path.join(dirpath, f)
    # Fallback: first parquet found anywhere
    for dirpath, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.parquet'):
                return os.path.join(dirpath, f)
    return None

PARQUET_PATH = find_parquet()
assert PARQUET_PATH, (
    'NuminaMath parquet not found in /kaggle/input.\n'
    'Check that nemotron-offline-deps dataset is attached and contains a .parquet file.'
)
print(f'Loading: {PARQUET_PATH}')

ds = load_dataset('parquet', data_files={'train': PARQUET_PATH}, split='train')
print(f'Loaded  : {len(ds):,} examples')
print(f'Columns : {ds.column_names}')
print(f'\nSample problem:\n{ds[0]["problem"][:300]}')
print(f'\nSample solution (first 200 chars):\n{ds[0]["solution"][:200]}')


## Cell 5 — Format dataset into chat template

In [ ]:
SYSTEM_PROMPT = (
    'You are a mathematical reasoning expert. '
    'Solve problems step by step, showing all working. '
    'Always place your final answer inside \\boxed{} at the end.'
)

def format_for_sft(example):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': example['problem']},
        {'role': 'assistant', 'content': example['solution']},
    ]
    return {
        'text': tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

# First 50k for SFT, remainder reserved for GRPO
N_SFT = min(50_000, len(ds))
sft_raw  = ds.select(range(N_SFT))
grpo_raw = ds.select(range(N_SFT, len(ds)))

print(f'SFT pool  : {len(sft_raw):,} examples')
print(f'GRPO pool : {len(grpo_raw):,} examples')

ds_fmt = sft_raw.map(
    format_for_sft,
    num_proc=2,
    remove_columns=sft_raw.column_names,
    desc='Formatting',
)
print(f'\nFormatted : {len(ds_fmt):,} examples')
print(f'\nSample (first 500 chars):')
print(ds_fmt[0]['text'][:500])


## Cell 6 — SFT Training

Expected duration: ~4–6 hrs on RTX Pro 6000 (48 GB VRAM).

**Fixes:**
- `eval_strategy=` → `evaluation_strategy=` (renamed in transformers ≥4.46; old key raises `TypeError`)
- `SFTTrainer(tokenizer=…)` → `processing_class=tokenizer` (trl ≥0.9 deprecation)


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

split    = ds_fmt.train_test_split(test_size=0.01, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print(f'Train : {len(train_ds):,}  |  Eval : {len(eval_ds):,}')

sft_args = SFTConfig(
    output_dir='/kaggle/working/checkpoints',

    # batch / gradient
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16
    per_device_eval_batch_size=2,

    # schedule
    num_train_epochs=2,
    warmup_ratio=0.05,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',

    # optimiser
    optim='adamw_8bit',
    weight_decay=0.01,
    max_grad_norm=1.0,

    # evaluation & checkpointing
    evaluation_strategy='steps',   # FIX 6: was eval_strategy (deprecated)
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # sequence / packing
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=True,

    # precision
    bf16=True,
    tf32=True,

    # misc
    logging_steps=25,
    report_to='none',
    seed=42,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,   # FIX 7: was tokenizer= (deprecated in trl ≥0.9)
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_args,
)

print(f'VRAM before training : {torch.cuda.memory_allocated() / 1e9:.1f} GB')
print('Starting SFT...')
trainer.train()
print('\nSFT complete!')


## Cell 7 — GRPO Reward Fine-tuning *(optional — run after SFT)*

Trains with two reward signals:
- **Correctness** (1.0): predicted `\boxed{}` matches gold answer exactly or numerically
- **Format** (0.2): response contains a `\boxed{}` expression at all

**Fixes:**
- `GRPOConfig(max_new_tokens=…)` → `max_completion_length=` (correct arg in trl ≥0.8)
- `GRPOConfig(temperature=…)` → moved into `generation_kwargs=` (temperature is a generation param, not a trainer param)
- `reward_correctness(…, solution, …)` → `solution=None` default so it doesn't crash when column is absent from a batch


In [ ]:
import re
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel

FastLanguageModel.for_training(model)

# ── reward helpers ─────────────────────────────────────────────────────────────
def extract_boxed(text: str) -> str:
    """Extract the content of the last \\boxed{} in text, handling nested braces."""
    m = re.search(r'\\boxed\{', text)
    if not m:
        return ''
    start, depth = m.end(), 1
    for i, ch in enumerate(text[start:]):
        depth += (ch == '{') - (ch == '}')
        if depth == 0:
            return text[start:start + i].strip()
    return ''

def num_eq(a: str, b: str, tol: float = 1e-6) -> bool:
    """True when a and b represent the same number within tolerance."""
    try:
        fa = float(a.replace(',', ''))
        fb = float(b.replace(',', ''))
        return abs(fa - fb) / max(abs(fb), 1e-9) < tol
    except ValueError:
        return False

def reward_correctness(completions, solution=None, **kwargs):
    """1.0 for correct answer, 0.1 for wrong but formatted, 0.0 for no box.
    FIX 10: solution=None default prevents TypeError when column absent from batch.
    """
    if solution is None:
        return [0.0] * len(completions)
    gold = extract_boxed(solution[0] if isinstance(solution, list) else solution)
    rewards = []
    for c in completions:
        pred = extract_boxed(c)
        if not pred:
            rewards.append(0.0)
        elif pred == gold or num_eq(pred, gold):
            rewards.append(1.0)
        else:
            rewards.append(0.1)
    return rewards

def reward_format(completions, **kwargs):
    """0.2 bonus just for using \\boxed{} — encourages the format even if wrong."""
    return [0.2 if re.search(r'\\boxed\{', c) else 0.0 for c in completions]

# ── format GRPO dataset ────────────────────────────────────────────────────────
def fmt_grpo(x):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': x['problem']},
    ]
    return {
        'prompt':   tokenizer.apply_chat_template(
                        msgs, tokenize=False, add_generation_prompt=True),
        'solution': x['solution'],
    }

grpo_ds = grpo_raw.map(fmt_grpo, remove_columns=grpo_raw.column_names, desc='GRPO format')
print(f'GRPO dataset: {len(grpo_ds):,} examples')

# ── GRPO config ────────────────────────────────────────────────────────────────
grpo_cfg = GRPOConfig(
    output_dir='/kaggle/working/grpo_checkpoints',

    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=0.3,
    warmup_ratio=0.1,

    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to='none',

    num_generations=4,
    max_completion_length=1024,                           # FIX 8: was max_new_tokens
    generation_kwargs={'temperature': 0.7, 'do_sample': True},  # FIX 9: was top-level arg
    seed=42,
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_correctness, reward_format],
    args=grpo_cfg,
    train_dataset=grpo_ds,
)

print('Starting GRPO...')
grpo_trainer.train()
print('\nGRPO complete!')


## Cell 8 — Save adapter & verify competition constraints

In [ ]:
import os, shutil, json

ADAPTER_OUT = '/kaggle/working/nemotron-adapter-ready-to-submit'
if os.path.exists(ADAPTER_OUT):
    shutil.rmtree(ADAPTER_OUT)
os.makedirs(ADAPTER_OUT)

model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f'Saved to: {ADAPTER_OUT}')

# ── verify adapter_config ─────────────────────────────────────────────────────
cfg_path = os.path.join(ADAPTER_OUT, 'adapter_config.json')
with open(cfg_path) as f:
    cfg = json.load(f)
print('\nadapter_config.json:')
print(json.dumps(cfg, indent=2))

# Competition hard limit: LoRA rank ≤ 32
rank = cfg.get('r', 999)
assert rank <= 32, (
    f'Rank {rank} exceeds competition limit of 32!\n'
    f'Retrain with LORA_RANK ≤ 32.'
)
print(f'\n✓ Rank check passed: r={rank} ≤ 32')

# File listing with sizes
print('\nFiles:')
total_mb = 0
for fn in sorted(os.listdir(ADAPTER_OUT)):
    sz = os.path.getsize(os.path.join(ADAPTER_OUT, fn)) / 1e6
    total_mb += sz
    print(f'  {fn:45s} {sz:6.1f} MB')
print(f'  {"TOTAL":45s} {total_mb:6.1f} MB')


## Cell 9 — Zip and submit

In [ ]:
import shutil, os

ZIP_BASE = '/kaggle/working/submission'
zip_path = shutil.make_archive(ZIP_BASE, 'zip', ADAPTER_OUT)
size_mb  = os.path.getsize(zip_path) / 1e6

print(f'Created : {zip_path}')
print(f'Size    : {size_mb:.1f} MB')
print('\nSubmit /kaggle/working/submission.zip to the competition.')
print('Done ✓')
